# Eliminación de texto en pósters — Google Colab

**Qué hace este notebook:**
1. Lee los pósters originales desde Google Drive (`posters/`)
2. Detecta regiones de texto con **EasyOCR** (GPU T4)
3. Elimina el texto con **inpainting** de OpenCV
4. Guarda los pósters limpios en Drive (`posters_clean/`)

**Antes de ejecutar:**
- Activar GPU: `Entorno de ejecución → Cambiar tipo → T4 GPU`
- Subir la carpeta `data/posters/` a tu Google Drive  
  (o ajustar `POSTERS_DIR` abajo)

## 1. Instalar dependencias

In [ ]:
!pip install -q easyocr opencv-python-headless tqdm

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuración

In [ ]:
from pathlib import Path

# ── AJUSTAR ESTAS RUTAS ────────────────────────────────────────────────────────
# Carpeta de Drive donde están los pósters originales
DRIVE_POSTERS_DIR = Path("/content/drive/MyDrive/recomendador_peliculas/data/posters")

# Carpeta de Drive donde se guardarán los pósters limpios al final
DRIVE_OUTPUT_DIR  = Path("/content/drive/MyDrive/recomendador_peliculas/data/posters_clean")

# Umbral de confianza mínimo para aceptar detecciones de texto (0-1)
MIN_CONFIDENCE = 0.4

# Píxeles de dilatación de la máscara (cubre bordes del texto)
DILATION_PX    = 8

# Reanudar procesamiento (omitir imágenes ya guardadas)
SKIP_EXISTING  = True
# ───────────────────────────────────────────────────────────────────────────────

# Rutas locales en Colab (SSD, mucho más rápidas que Drive durante el procesamiento)
LOCAL_POSTERS_DIR = Path("/content/posters")
LOCAL_OUTPUT_DIR  = Path("/content/posters_clean")
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Pósters de entrada (Drive) : {DRIVE_POSTERS_DIR}")
print(f"Salida final (Drive)       : {DRIVE_OUTPUT_DIR}")


## 4. Verificar GPU

In [ ]:
import torch

USE_GPU = torch.cuda.is_available()
device_name = torch.cuda.get_device_name(0) if USE_GPU else "CPU"
print(f"Dispositivo: {device_name}")

if not USE_GPU:
    print("⚠️  No se detectó GPU. El procesamiento será muy lento.")
    print("   Ve a Entorno de ejecución → Cambiar tipo de entorno → T4 GPU")

## 5. Funciones de procesamiento

In [ ]:
import cv2
import numpy as np


def build_text_mask(image_bgr: np.ndarray, results: list, dilation_px: int) -> np.ndarray:
    """Construye una máscara binaria con las regiones de texto detectadas."""
    h, w = image_bgr.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)

    for bbox, _text, _conf in results:
        pts = np.array(bbox, dtype=np.int32)
        cv2.fillPoly(mask, [pts], 255)

    if dilation_px > 0:
        kernel = cv2.getStructuringElement(
            cv2.MORPH_RECT, (dilation_px * 2 + 1, dilation_px * 2 + 1)
        )
        mask = cv2.dilate(mask, kernel, iterations=1)

    return mask


def remove_text(image_path: Path, reader, dilation_px: int, min_confidence: float):
    """Elimina el texto de una imagen mediante EasyOCR + inpainting.
    Devuelve la imagen procesada en BGR, o None si hubo error de lectura.
    """
    img_bgr = cv2.imread(str(image_path))
    if img_bgr is None:
        return None

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    try:
        results = reader.readtext(img_rgb, detail=1)
    except Exception:
        return img_bgr

    results = [(bbox, text, conf) for bbox, text, conf in results if conf >= min_confidence]

    if not results:
        return img_bgr

    mask = build_text_mask(img_bgr, results, dilation_px)
    return cv2.inpaint(img_bgr, mask, inpaintRadius=3, flags=cv2.INPAINT_TELEA)


print("Funciones definidas.")

## 6. Inicializar EasyOCR

In [ ]:
import easyocr

print("Inicializando EasyOCR (descarga el modelo la primera vez ~100 MB)...")
reader = easyocr.Reader(["en"], gpu=USE_GPU, verbose=False)
print("EasyOCR listo.")

## 7. Vista previa (opcional)
Verifica el resultado en un póster de muestra antes de procesar todo el dataset.

In [ ]:
import matplotlib.pyplot as plt

sample_paths = sorted(DRIVE_POSTERS_DIR.glob("*.jpg"))[:1]

if not sample_paths:
    print("No se encontraron imágenes en", DRIVE_POSTERS_DIR)
else:
    sample = sample_paths[0]
    original_bgr = cv2.imread(str(sample))
    processed_bgr = remove_text(sample, reader, DILATION_PX, MIN_CONFIDENCE)

    fig, axes = plt.subplots(1, 2, figsize=(10, 7))
    axes[0].imshow(cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Original")
    axes[0].axis("off")
    axes[1].imshow(cv2.cvtColor(processed_bgr, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Sin texto")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()
    print(f"Muestra: {sample.name}")


## 8. Copiar pósters a almacenamiento local y procesar

Trabajar en `/content/` (SSD local de Colab) es ~5× más rápido que leer/escribir
directamente desde Google Drive. Al terminar, los resultados se sincronizan a Drive.

In [ ]:
import shutil
import time
from tqdm.notebook import tqdm

# ── PASO 1: copiar pósters de Drive a almacenamiento local ──────────────────────
print("Copiando pósters de Drive a /content/posters ...")
t0 = time.time()

if not LOCAL_POSTERS_DIR.exists() or not any(LOCAL_POSTERS_DIR.iterdir()):
    shutil.copytree(str(DRIVE_POSTERS_DIR), str(LOCAL_POSTERS_DIR))
else:
    print("  Ya existen en /content/posters, omitiendo copia.")

n_posters = len(list(LOCAL_POSTERS_DIR.glob("*.jpg")))
print(f"  {n_posters} pósters listos en {time.time()-t0:.0f}s")

# ── PASO 2: procesar en almacenamiento local ────────────────────────────────────
print("\nProcesando...")
image_paths = sorted(LOCAL_POSTERS_DIR.glob("*.jpg")) + sorted(LOCAL_POSTERS_DIR.glob("*.png"))

if not image_paths:
    raise FileNotFoundError(f"No se encontraron imágenes en {LOCAL_POSTERS_DIR}")

processed    = 0
text_removed = 0
skipped      = 0
errors       = 0
t_start = time.time()

for img_path in tqdm(image_paths, desc="Eliminando texto", unit="img"):
    out_path = LOCAL_OUTPUT_DIR / img_path.name

    if SKIP_EXISTING and out_path.exists():
        skipped += 1
        continue

    result = remove_text(img_path, reader, DILATION_PX, MIN_CONFIDENCE)

    if result is None:
        shutil.copy2(img_path, out_path)
        errors += 1
        continue

    original = cv2.imread(str(img_path))
    if original is not None and not np.array_equal(result, original):
        text_removed += 1

    cv2.imwrite(str(out_path), result)
    processed += 1

elapsed = time.time() - t_start
rate    = processed / elapsed if elapsed > 0 else 0

print(f"""
Procesamiento completado en {elapsed/60:.1f} min
   Procesadas       : {processed}  ({rate:.2f} img/s)
   Con texto borrado: {text_removed}
   Omitidas         : {skipped}
   Errores lectura  : {errors}
""")

# ── PASO 3: sincronizar resultados a Drive ──────────────────────────────────────
print("Subiendo resultados a Drive...")
t0 = time.time()
shutil.copytree(str(LOCAL_OUTPUT_DIR), str(DRIVE_OUTPUT_DIR), dirs_exist_ok=True)
print(f"  Sincronización completada en {time.time()-t0:.0f}s → {DRIVE_OUTPUT_DIR}")


## 9. (Opcional) Comprimir y descargar los pósters limpios
Solo necesario si prefieres descargarlos en lugar de dejarlos en Drive.

In [ ]:
# Descomenta para comprimir y descargar

# import shutil
# from google.colab import files
#
# zip_path = "/content/posters_clean.zip"
# shutil.make_archive("/content/posters_clean", "zip", OUTPUT_DIR)
# print(f"ZIP creado: {zip_path}")
# files.download(zip_path)